# Quantum Machine Learning on the SuperCon Tc Dataset
**Goal:** predict the critical temperature (Tc) of superconductors with two quantum models, and compare them fairly against classical models, including the XGBoost model from my undergraduate project.

**Roadmap**
| Step | What happens |
|---|---|
| 1 | Install and import libraries |
| 2 | Choose settings (number of qubits, samples, etc.) |
| 3 | Load the featurized data from the XGBoost project |
| 4 | Take a smaller subset and split into train / test |
| 5 | Shrink 132 features to a few with PCA, convert to angles |
| 6 | Classical baselines (XGBoost, SVR) |
| 7 | Quantum model 1: Quantum kernel + SVR |
| 8 | Quantum model 2: Variational quantum regressor |
| 9 | Compare all five models |
| 10 | Repeat over several splits and qubit numbers, test significance |

Run the cells **in order, top to bottom**.

## Step 1: Install and import libraries
- **PennyLane** is the quantum ML library. It lets us build quantum circuits and run them on a *simulator* (a normal computer pretending to be a quantum computer).
- The other libraries are the same ones from the XGBoost project.

In [ ]:
# Install PennyLane (the other libraries are already in Colab, but listing them is harmless)
!pip install -q pennylane scikit-learn xgboost pandas numpy

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Quantum library
import pennylane as qml
from pennylane import numpy as pnp   # PennyLane's version of numpy: it can compute gradients
                                     # (needed later to TRAIN the quantum circuit)

# Classical ML tools (same as the XGBoost project)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

print("PennyLane version:", qml.__version__)

## Step 2: Settings
These are the "knobs" you will experiment with later.

**Why so small?** A simulator stores 2ⁿ numbers for n qubits. Each extra qubit **doubles** the cost:
6 qubits → 64 numbers, 10 qubits → 1,024, 20 qubits → over a million.
In this notebook **1 qubit holds 1 feature**, so we must use only a few features and a few hundred materials.

In [ ]:
N_QUBITS  = 6     # number of qubits = number of features fed to the quantum models (try 4, 6, 8)
N_SAMPLES = 600   # how many materials to use in total (train + test). The full set is ~21,000
N_LAYERS  = 3     # depth of the trainable quantum circuit in Step 8 (more layers = more parameters)
EPOCHS    = 40    # how many times the variational model sees the whole training set
SEED      = 42    # fixes randomness so results are repeatable

rng = np.random.default_rng(SEED)   # random number generator used for sampling and shuffling

## Step 3: Load the featurized data
This is the `featurized_data.csv` saved by the XGBoost project: each row is a material, the columns are 132 Magpie features plus `critical_temp`. Upload it first with the folder icon on the left.

In [ ]:
df = pd.read_csv("featurized_data.csv")

# X = the inputs (features). Drop the text columns, keep only numbers.
X_all = df.drop(columns=["material", "critical_temp", "composition"], errors="ignore")
X_all = X_all.select_dtypes("number")

# y = the target we want to predict
y_all = df["critical_temp"].values

# Some formulas may have failed in Magpie featurization and contain empty values (NaN). Remove them.
good_rows = ~X_all.isna().any(axis=1)
X_all, y_all = X_all[good_rows].values, y_all[good_rows]

print(f"Materials: {X_all.shape[0]}   Features per material: {X_all.shape[1]}")
print(f"Tc range: {y_all.min():.1f} K to {y_all.max():.1f} K")

## Step 4: Take a random subset and split it
- Pick `N_SAMPLES` materials at random (to keep the quantum simulation fast).
- Split them: **75% for training**, **25% for testing**. The test set is never seen during training, so it gives an honest score.

In [ ]:
# Randomly choose which materials to keep (no repeats)
idx = rng.choice(len(y_all), size=min(N_SAMPLES, len(y_all)), replace=False)
X, y = X_all[idx], y_all[idx]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED)

print("Training materials:", len(y_train))
print("Test materials:    ", len(y_test))

## Step 5: Prepare the data for a quantum circuit
Three transformations, done in this order:

1. **Standardize** each feature to mean 0, standard deviation 1. PCA needs this so that large-valued features (like atomic weight) don't dominate.
2. **PCA** compresses the 132 Magpie features into `N_QUBITS` new features that keep as much information as possible. Magpie features are strongly correlated, so a few components keep a lot of the information.
3. **Rescale to angles between 0 and π.** Quantum circuits take data in as **rotation angles** of qubits, so each feature must become an angle.

⚠️ Every transformation is **fitted on the training set only** and then applied to the test set. Fitting on everything would leak test information into training.

In [ ]:
# 1. Standardize (learn mean and std from the training data)
scaler = StandardScaler().fit(X_train)

# 2. PCA: learn the N_QUBITS most informative directions from the training data
pca = PCA(n_components=N_QUBITS, random_state=SEED).fit(scaler.transform(X_train))

# 3. Rescale the PCA outputs into angles in [0, pi]
angle_scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(
    pca.transform(scaler.transform(X_train)))

def to_angles(data):
    """Raw Magpie features -> standardized -> PCA -> angles in [0, pi]."""
    a = angle_scaler.transform(pca.transform(scaler.transform(data)))
    return np.clip(a, 0, np.pi)   # test values can fall slightly outside the range, so clip them

A_train = to_angles(X_train)   # quantum-ready training inputs, shape (n_train, N_QUBITS)
A_test  = to_angles(X_test)    # quantum-ready test inputs

print(f"PCA kept {pca.explained_variance_ratio_.sum():.1%} of the information "
      f"using {N_QUBITS} features instead of {X.shape[1]}")
print("First training material as angles:", np.round(A_train[0], 2))

**Scaling the target too.** The variational model (Step 8) outputs a quantum measurement, which is **always between −1 and +1**. So we squeeze Tc into [−1, 1] for training, and convert predictions back to Kelvin afterwards.

In [ ]:
y_scaler = MinMaxScaler(feature_range=(-1, 1)).fit(y_train.reshape(-1, 1))
y_train_scaled = y_scaler.transform(y_train.reshape(-1, 1)).ravel()

print("Original Tc of first 3:", np.round(y_train[:3], 1))
print("Scaled to [-1, 1]:     ", np.round(y_train_scaled[:3], 3))

A small helper so every model is scored the same way (the same R² and RMSE you used in the project):

In [ ]:
results = {}   # stores the score of every model for the final comparison

def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    results[name] = {"R²": r2, "RMSE (K)": rmse, "pred": y_pred}
    print(f"{name:<34} R² = {r2:6.3f}   RMSE = {rmse:6.2f} K")

## Step 6: Classical baselines
To judge the quantum models fairly, we need classical models trained on the **same materials**:

| Model | Features | Why it is here |
|---|---|---|
| XGBoost | all 132 | The project model: the best we could hope for |
| XGBoost | the 6 PCA features | Shows what is lost by compressing to 6 features |
| SVR (RBF kernel) | the 6 PCA features | The direct rival of the quantum kernel in Step 7 |

In [ ]:
# Model 1: your original project XGBoost, same hyperparameters, all 132 features
xgb_full = XGBRegressor(n_estimators=374, learning_rate=0.02, max_depth=16,
                        min_child_weight=1, colsample_bytree=0.5, random_state=1)
xgb_full.fit(X_train, y_train)
evaluate("1. XGBoost (132 features)", y_test, xgb_full.predict(X_test))

# Model 2: XGBoost on the same 6 PCA features the quantum models get
# (max_depth lowered to 6 because there are far fewer features now)
xgb_pca = XGBRegressor(n_estimators=374, learning_rate=0.02, max_depth=6, random_state=1)
xgb_pca.fit(A_train, y_train)
evaluate(f"2. XGBoost ({N_QUBITS} PCA features)", y_test, xgb_pca.predict(A_test))

# Model 3: classical Support Vector Regression with the standard RBF kernel
svr_rbf = SVR(kernel="rbf", C=50)
svr_rbf.fit(A_train, y_train)
evaluate(f"3. Classical SVR ({N_QUBITS} PCA)", y_test, svr_rbf.predict(A_test))

## Step 7: Quantum model 1, Quantum kernel + SVR

**The idea in plain words**
- An SVR doesn't look at materials directly. It only needs a **similarity score** between every pair of materials (the *kernel*).
- A classical RBF kernel measures similarity with a distance formula.
- A **quantum kernel** instead:
  1. encodes each material into a quantum state |ψ(x)⟩ using a circuit (the *feature map*),
  2. measures how much two states overlap: similarity = |⟨ψ(x)|ψ(x′)⟩|². Identical materials give 1; very different ones give close to 0.
- The SVR itself is completely classical. **Only the similarity measure is quantum.**

### 7a. The feature map: how a material becomes a quantum state
Gates used:
- **Hadamard (H):** puts a qubit into superposition (equal mix of 0 and 1).
- **RZ(angle):** rotates the qubit by an angle. This is where a feature value enters.
- **CNOT – RZ – CNOT:** entangles two neighbouring qubits, with a rotation depending on the **product** of their two features. This lets the circuit capture interactions between features.

In [ ]:
dev_kernel = qml.device("default.qubit", wires=N_QUBITS)   # a simulated quantum computer

@qml.qnode(dev_kernel)          # turns the Python function into a runnable quantum circuit
def feature_map(x):
    """Encode one material (N_QUBITS angles) into a quantum state."""
    for repeat in range(2):                     # repeat the encoding twice to make it richer
        # (i) single-qubit encoding: one feature per qubit
        for q in range(N_QUBITS):
            qml.Hadamard(wires=q)               # superposition
            qml.RZ(x[q], wires=q)               # rotate by the feature value
        # (ii) entangle neighbouring qubits using the product of their features
        for q in range(N_QUBITS - 1):
            qml.CNOT(wires=[q, q + 1])
            qml.RZ(x[q] * x[q + 1] / np.pi, wires=q + 1)
            qml.CNOT(wires=[q, q + 1])
    return qml.state()   # return the full quantum state (possible only on a simulator)

# Draw the circuit for the first training material
print(qml.draw(feature_map, decimals=2)(A_train[0]))

### 7b. Build the kernel (similarity) matrices
- `K_train[i, j]` = similarity between training materials i and j → used to **fit** the SVR.
- `K_test[i, j]` = similarity between test material i and training material j → used to **predict**.

Shortcut: we compute each material's quantum state **once**, then get all overlaps with one matrix multiplication. That is much faster than running a separate circuit for every pair.

In [ ]:
def quantum_kernel(A, B):
    states_A = np.array([feature_map(a) for a in A])   # one quantum state per material
    states_B = np.array([feature_map(b) for b in B])
    overlaps = states_A.conj() @ states_B.T            # <psi(a)|psi(b)> for every pair
    return np.abs(overlaps) ** 2                       # similarity = |overlap|^2, between 0 and 1

t0 = time.time()
K_train = quantum_kernel(A_train, A_train)   # shape (n_train, n_train)
K_test  = quantum_kernel(A_test,  A_train)   # shape (n_test,  n_train)
print(f"Kernel matrices built in {time.time() - t0:.1f} s")
print("Similarity of material 0 with itself:", round(K_train[0, 0], 3), "(should be 1)")
print("Similarity of material 0 with material 1:", round(K_train[0, 1], 3))

### 7c. Train the SVR on the quantum kernel
`kernel="precomputed"` tells scikit-learn: *don't compute similarities yourself, use my quantum ones.* Everything else matches Model 3, so any difference in score comes from the kernel alone.

In [ ]:
qsvr = SVR(kernel="precomputed", C=50)
qsvr.fit(K_train, y_train)
evaluate("4. Quantum kernel SVR", y_test, qsvr.predict(K_test))

## Step 8: Quantum model 2, Variational quantum regressor ("quantum neural network")

**The idea in plain words**
- Build a circuit with **trainable rotation angles**. These play the role of weights in a neural network.
- Feed in a material, measure qubit 0, and read a number between −1 and +1. That number is the (scaled) prediction.
- Compare the prediction with the true Tc and **adjust the angles with gradient descent** to reduce the error. Repeat.

### 8a. The circuit
Each layer has two parts:
1. **AngleEmbedding:** loads the 6 features as Y-rotations (the data goes in).
2. **StronglyEntanglingLayers:** trainable rotations on every qubit, then CNOTs linking them (the learning part).

We load the data again in **every layer** ("data re-uploading"). This makes the model much more expressive, a bit like adding hidden layers.

In [ ]:
dev_vqc = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev_vqc)
def vqc(x, weights):
    for layer in range(N_LAYERS):
        qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation="Y")             # data in
        qml.StronglyEntanglingLayers(weights[layer:layer + 1], wires=range(N_QUBITS))  # trainable part
    return qml.expval(qml.PauliZ(0))   # measure qubit 0: average value between -1 and +1

# Trainable parameters: 3 rotation angles per qubit per layer
weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=N_LAYERS, n_wires=N_QUBITS)
print("Weight tensor shape (layers, qubits, 3 angles):", weight_shape)

# Draw the circuit with random example weights
example_w = np.random.uniform(0, 2 * np.pi, size=weight_shape)
print(qml.draw(vqc, decimals=1, level="device")(A_train[0], example_w))

### 8b. Prediction and loss functions
- `predict` runs the circuit for a **whole batch** of materials at once (PennyLane's "broadcasting"), then adds a trainable **bias** (an offset, like the intercept in linear regression).
- `loss` is the **mean squared error**, the quantity we want to make small.

In [ ]:
def predict(weights, bias, A):
    A = pnp.array(A, requires_grad=False)   # the data is fixed: we don't train it
    return vqc(A, weights) + bias

def loss(weights, bias, A, y_scaled):
    return pnp.mean((predict(weights, bias, A) - y_scaled) ** 2)

### 8c. Train with gradient descent
- Start from **random** angles.
- **Adam** optimizer (same as in deep learning): for each mini-batch of 32 materials it computes how the loss changes with every angle (the gradient), and nudges the angles downhill.
- Watch the training loss fall. If it stops falling, the model has learned what it can.

In [ ]:
pnp.random.seed(SEED)
weights = pnp.random.uniform(0, 2 * np.pi, size=weight_shape, requires_grad=True)  # random start
bias = pnp.array(0.0, requires_grad=True)
optimizer = qml.AdamOptimizer(stepsize=0.05)   # stepsize = learning rate

BATCH_SIZE = 32
loss_history = []
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    order = rng.permutation(len(A_train))                # shuffle materials each epoch
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start:start + BATCH_SIZE]
        # One optimisation step: compute the gradient on this batch, update weights and bias
        (weights, bias, _, _), _ = optimizer.step_and_cost(
            loss, weights, bias, A_train[batch], y_train_scaled[batch])

    epoch_loss = float(loss(weights, bias, A_train, y_train_scaled))
    loss_history.append(epoch_loss)
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d}   training loss = {epoch_loss:.4f}")

print(f"Training took {time.time() - t0:.0f} s for {weights.size + 1} trainable parameters")

plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch"); plt.ylabel("Training MSE (scaled Tc)")
plt.title("Variational quantum regressor: learning curve"); plt.show()

### 8d. Evaluate on the test set (convert predictions back to Kelvin first)

In [ ]:
pred_scaled = np.array(predict(weights, bias, A_test))
pred_kelvin = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
evaluate("5. Variational quantum regressor", y_test, pred_kelvin)

## Step 9: Compare all five models
How to read this:
- **Model 3 vs 4:** same SVR, same features, only the kernel differs → the cleanest test of the quantum kernel.
- **Model 2 vs 5:** classical vs quantum model learning from the same 6 features.
- **Model 1:** the ceiling, showing how much information PCA compression costs.

In [ ]:
table = pd.DataFrame({k: {m: v[m] for m in ["R²", "RMSE (K)"]} for k, v in results.items()}).T
print(table.round(3))

# Predicted vs actual Tc for every model: points on the dashed line are perfect predictions
fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4), sharex=True, sharey=True)
for ax, (name, r) in zip(axes, results.items()):
    ax.scatter(y_test, r["pred"], s=10, alpha=0.6)
    lim = [0, max(y_test.max(), np.max(r["pred"]))]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_title(f"{name}\nR² = {r['R²']:.2f}", fontsize=9)
    ax.set_xlabel("Actual Tc (K)")
axes[0].set_ylabel("Predicted Tc (K)")
plt.tight_layout(); plt.show()

## Step 10: Is the difference real? Repeat over several random splits
One test set of 150 materials is noisy: a different random subset could reverse the ranking.
This step reruns models 2, 3 and 4 for **4, 6 and 8 qubits**, each with **5 different random subsets**, and reports:
- the average R² and its spread (standard deviation),
- how often the quantum kernel beat the classical SVR **on the same split**.

Overlapping error bars mean the models are effectively tied. Takes a few minutes.

In [ ]:
# ===== STEP 10: Repeat the comparison over several random seeds and qubit numbers =====
# Why: one train/test split of 150 materials is noisy. Repeating with different
# random subsets tells us whether a difference between models is real or just luck.

QUBIT_LIST = [4, 6, 8]          # numbers of qubits (= PCA features) to compare
SEED_LIST  = [1, 2, 3, 4, 5]    # 5 different random subsets / splits
MODELS     = ["XGBoost (PCA)", "Classical SVR", "Quantum kernel SVR"]

def make_kernel_fn(n_qubits):
    """Build the same quantum feature map as Step 7a, for any number of qubits."""
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def fmap(x):
        for repeat in range(2):
            for q in range(n_qubits):
                qml.Hadamard(wires=q)
                qml.RZ(x[q], wires=q)
            for q in range(n_qubits - 1):
                qml.CNOT(wires=[q, q + 1])
                qml.RZ(x[q] * x[q + 1] / np.pi, wires=q + 1)
                qml.CNOT(wires=[q, q + 1])
        return qml.state()

    def kernel(A, B):
        SA = np.array([fmap(a) for a in A])
        SB = np.array([fmap(b) for b in B])
        return np.abs(SA.conj() @ SB.T) ** 2
    return kernel

def run_once(n_qubits, seed):
    """Steps 4-7 in one function: subset, split, PCA->angles, train models 2, 3, 4."""
    r = np.random.default_rng(seed)
    idx = r.choice(len(y_all), size=min(N_SAMPLES, len(y_all)), replace=False)
    Xtr, Xte, ytr, yte = train_test_split(X_all[idx], y_all[idx],
                                          test_size=0.25, random_state=seed)
    # Preprocess exactly as in Step 5 (fitted on training data only)
    sc = StandardScaler().fit(Xtr)
    pc = PCA(n_components=n_qubits, random_state=seed).fit(sc.transform(Xtr))
    ang = MinMaxScaler((0, np.pi)).fit(pc.transform(sc.transform(Xtr)))
    f = lambda D: np.clip(ang.transform(pc.transform(sc.transform(D))), 0, np.pi)
    Atr, Ate = f(Xtr), f(Xte)

    scores = {}
    m = XGBRegressor(n_estimators=374, learning_rate=0.02, max_depth=6,
                     random_state=1).fit(Atr, ytr)
    scores["XGBoost (PCA)"] = r2_score(yte, m.predict(Ate))
    m = SVR(kernel="rbf", C=50).fit(Atr, ytr)
    scores["Classical SVR"] = r2_score(yte, m.predict(Ate))
    kernel = make_kernel_fn(n_qubits)
    m = SVR(kernel="precomputed", C=50).fit(kernel(Atr, Atr), ytr)
    scores["Quantum kernel SVR"] = r2_score(yte, m.predict(kernel(Ate, Atr)))
    scores["PCA info kept"] = pc.explained_variance_ratio_.sum()
    return scores

# Run every combination and collect the R² scores
rows = []
t0 = time.time()
for nq in QUBIT_LIST:
    for s in SEED_LIST:
        res = run_once(nq, s)
        rows.append({"qubits": nq, "seed": s, **res})
        print(f"qubits={nq} seed={s}  " +
              "  ".join(f"{k}={v:.3f}" for k, v in res.items()))
print(f"\nFinished in {time.time() - t0:.0f} s")

runs = pd.DataFrame(rows)

# Summary table: mean ± standard deviation of R² for each model and qubit number
g = runs.groupby("qubits")
summary = pd.DataFrame({m: g[m].mean().map("{:.3f}".format) + " ± " + g[m].std().map("{:.3f}".format)
                        for m in MODELS})
summary["PCA info kept"] = g["PCA info kept"].mean().map("{:.1%}".format)
print("\nTest R² averaged over", len(SEED_LIST), "random splits (mean ± std):")
print(summary.to_string())

# How often did the quantum kernel beat the classical SVR on the same split?
runs["quantum wins"] = runs["Quantum kernel SVR"] > runs["Classical SVR"]
print(f"\nQuantum kernel beat classical SVR in this many of {len(SEED_LIST)} splits:")
print(runs.groupby("qubits")["quantum wins"].sum().to_string())

# Plot: mean R² with error bars (± 1 std) for each model vs number of qubits
for name in MODELS:
    plt.errorbar(QUBIT_LIST, g[name].mean(), yerr=g[name].std(), marker="o", capsize=4, label=name)
plt.xlabel("Number of qubits (= PCA features)"); plt.ylabel("Test R²")
plt.title(f"Average over {len(SEED_LIST)} random splits"); plt.legend(); plt.show()

### Paired significance test
A paired t-test asks: across all splits, is the quantum-minus-classical difference reliably different from zero?
A p-value above 0.05 means there is no evidence that one kernel is better than the other.

In [ ]:
from scipy.stats import ttest_rel

diff = runs["Quantum kernel SVR"] - runs["Classical SVR"]
t_stat, p_value = ttest_rel(runs["Quantum kernel SVR"], runs["Classical SVR"])
print(f"Mean R² difference (quantum - classical): {diff.mean():+.4f}")
print(f"Quantum kernel better in {(diff > 0).sum()} of {len(diff)} paired splits")
print(f"Paired t-test: t = {t_stat:.2f}, p = {p_value:.3f}")

## Experiments to try next
Change **one** setting in Step 2, then *Runtime → Run all*, and note how the scores and run times change:
1. `N_QUBITS` = 4, then 8. Does more information help, or does the model become harder to train?
2. `N_LAYERS` = 1, then 5 for the variational model.
3. In Step 7a, change `range(2)` to `range(1)` or `range(3)`, or remove the entangling part entirely. How much does entanglement matter?
4. `N_SAMPLES` = 1000 (slower, but more data).